# B++ Fine-tuning — Qwen2.5-3B-Instruct — FINAL 10K Run
**Project:** Automated Code Review & Optimization for LLM-Generated Code

## ⚠️ Before You Run
1. Add `HF_TOKEN` as a **Kaggle Secret** (not plain text)
2. Datasets attached: `dataset10k` + `staticAnalysis`
3. GPU: Tesla T4 — enabled
4. **Never restart session mid-training**
5. Run cells **in order, top to bottom**

## Cell Map
| Cell | Purpose |
|------|---------|
| 1 | Config — edit ONLY this cell |
| 2 | Install dependencies |
| 3 | Verify GPU + environment |
| 4 | Analyze dataset → compute max_seq_len |
| 5 | Load model + LoRA |
| 6 | Load + format dataset |
| 7 | Training |
| 8 | Save adapters + merge model |
| 9 | Inference test — sanity check |
| 10 | Upload to HuggingFace |
| 11 | Final summary |

In [12]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — CONFIGURATION  (ONLY EDIT THIS CELL)
# ═══════════════════════════════════════════════════════════
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# ───── Dataset ─────

DATA_PATH      = "/kaggle/input/datasets/sasaerreer/scorevdata/scorev_clean_full.jsonl"
STATIC_DIR     = '/kaggle/input/datasets/sasaerreer/static-analysis-engine'
SAMPLE_SIZE    = 8761                       

# ───── Run identity ─────
RUN_NAME       = 'qwen_bpp_v2_10k'
HF_REPO_ID     = 'sarahjaradat2004/qwen-scorev-v2'

# ───── Model ─────
BASE_MODEL     = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN    = 4096

# ───── QLoRA ─────
LORA_R         = 16                          # 16 for final 10K run
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.0                         # MUST be 0.0 with Unsloth
LORA_TARGETS   = ['q_proj','k_proj','v_proj','o_proj',
                   'gate_proj','up_proj','down_proj']

# ───── Training ─────
NUM_EPOCHS     = 1
BATCH_SIZE     = 1                           # T4: must be 1
GRAD_ACCUM     = 8                           # effective batch = 8
LR             = 2e-4
LR_SCHEDULER   = 'cosine'
WARMUP_STEPS   = 10                          # NOT warmup_ratio
MAX_GRAD_NORM  = 0.3

# ───── Paths ─────
OUTPUT_DIR     = f'/kaggle/working/{RUN_NAME}'
ADAPTERS_DIR   = f'{OUTPUT_DIR}/adapters'
MERGED_DIR     = f'{OUTPUT_DIR}/merged_model'

# ═══════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ═══════════════════════════════════════════════════════════
os.makedirs(OUTPUT_DIR,   exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
os.makedirs(MERGED_DIR,   exist_ok=True)

print('✅ Config loaded')
print(f'   Dataset     : {DATA_PATH}')
print(f'   Samples     : {SAMPLE_SIZE}')
print(f'   Model       : {BASE_MODEL}')
print(f'   LoRA        : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'   Epochs      : {NUM_EPOCHS}')
print(f'   Eff. Batch  : {BATCH_SIZE * GRAD_ACCUM}')
print(f'   Max seq len : {MAX_SEQ_LEN}')
print(f'   Output      : {OUTPUT_DIR}')

✅ Config loaded
   Dataset     : /kaggle/input/datasets/sasaerreer/scorevdata/scorev_clean_full.jsonl
   Samples     : 8761
   Model       : Qwen/Qwen2.5-3B-Instruct
   LoRA        : r=16, alpha=16, dropout=0.0
   Epochs      : 1
   Eff. Batch  : 8
   Max seq len : 4096
   Output      : /kaggle/working/qwen_bpp_v2_10k


In [3]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Install dependencies (Colab)
# ═══════════════════════════════════════════════════════════
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'⚠️  FAILED: {cmd}\n{result.stderr[-800:]}')
    else:
        print(f'✅ {cmd}')

# Single resolver pass: let unsloth pull a mutually-compatible stack.
# Pinned versions = reproducible + won't break if unsloth ships a new release tomorrow.
run('pip install -q '
    '"unsloth==2026.6.1" "unsloth_zoo" '
    '"trl>=0.18.2,<=0.24.0" "peft>=0.18.0" "accelerate>=0.34.1"')

# Verify the stack actually loaded with compatible versions
print('\n🔍 Installed versions:')
import importlib
for pkg in ['torch', 'transformers', 'trl', 'peft', 'accelerate', 'unsloth']:
    try:
        m = importlib.import_module(pkg)
        print(f'   {pkg:<14} {getattr(m, "__version__", "?")}')
    except Exception as e:
        print(f'   {pkg:<14} ❌ not importable: {e}')

print('\n✅ Dependencies ready')

✅ pip install -q "unsloth==2026.6.1" "unsloth_zoo" "trl>=0.18.2,<=0.24.0" "peft>=0.18.0" "accelerate>=0.34.1"

🔍 Installed versions:
   torch          2.10.0+cu128
   transformers   5.5.0
   trl            0.24.0


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


   peft           0.18.1
   accelerate     1.12.0
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:165: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
   unsloth        2026.6.1

✅ Dependencies ready


In [4]:
# DIAGNOSTIC 1: version check — السبب الجذري للـ PicklingError
import trl, transformers, unsloth, peft, torch
print('trl         :', trl.__version__)
print('transformers:', transformers.__version__)
print('unsloth     :', unsloth.__version__)
print('peft        :', peft.__version__)

# هل SFTConfig هو نفسه أم patched (مصدر الـ crash)؟
from trl import SFTConfig
import trl.trainer.sft_config as m
print('SFTConfig identity OK:', SFTConfig is m.SFTConfig)
print('  ↑ False = هذا سبب الـ PicklingError')

trl         : 0.24.0
transformers: 5.5.0
unsloth     : 2026.6.1
peft        : 0.18.1
SFTConfig identity OK: True
  ↑ False = هذا سبب الـ PicklingError


In [5]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Verify GPU + environment
# ═══════════════════════════════════════════════════════════
import torch
import transformers, unsloth, trl, peft

print('─── GPU ───')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram = gpu.total_memory / 1024**3
    print(f'✅ {gpu.name} — {vram:.1f} GB VRAM')
    if vram < 14:
        print('⚠️  Less than 14GB — may OOM. Consider reducing LORA_R to 8.')
else:
    print('❌ No GPU detected — check Kaggle accelerator setting!')
    raise RuntimeError('GPU required')

print('\n─── Versions ───')
print(f'  torch         : {torch.__version__}')
print(f'  transformers  : {transformers.__version__}')
print(f'  unsloth       : {unsloth.__version__}')
print(f'  trl           : {trl.__version__}')
print(f'  peft          : {peft.__version__}')

print('\n─── Dataset files ───')
import os
if os.path.exists(DATA_PATH):
    size_mb = os.path.getsize(DATA_PATH) / 1024**2
    print(f'✅ {DATA_PATH} ({size_mb:.1f} MB)')
else:
    print(f'❌ Dataset not found: {DATA_PATH}')
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

# Find static engine
static_path = None
for root, dirs, files in os.walk(STATIC_DIR):
    for f in files:
        if f.endswith('.py'):
            static_path = os.path.join(root, f)
            print(f'✅ Static Engine: {static_path}')
            break

if static_path is None:
    print(f'❌ Static Engine .py not found in {STATIC_DIR}')
    raise FileNotFoundError('Static Engine not found')

print('\n✅ Environment verified')

─── GPU ───
✅ Tesla T4 — 14.6 GB VRAM

─── Versions ───
  torch         : 2.10.0+cu128
  transformers  : 5.5.0
  unsloth       : 2026.6.1
  trl           : 0.24.0
  peft          : 0.18.1

─── Dataset files ───
✅ /kaggle/input/datasets/sasaerreer/scorevdata/scorev_clean_full.jsonl (70.6 MB)
✅ Static Engine: /kaggle/input/datasets/sasaerreer/static-analysis-engine/static_analysis_engine.py

✅ Environment verified


In [6]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Analyze dataset → token length distribution
# ═══════════════════════════════════════════════════════════
import json
import random
from transformers import AutoTokenizer

print('Loading tokenizer for analysis...')
tokenizer_analysis = AutoTokenizer.from_pretrained(
    BASE_MODEL, trust_remote_code=True
)

# Load samples
all_samples = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                all_samples.append(json.loads(line))
            except json.JSONDecodeError:
                pass

print(f'Total samples in file: {len(all_samples):,}')

# Sample
random.seed(42)
if len(all_samples) > SAMPLE_SIZE:
    samples = random.sample(all_samples, SAMPLE_SIZE)
else:
    samples = all_samples

print(f'Using: {len(samples):,} samples')

# Token length analysis
lengths = []
for s in samples:
    text = tokenizer_analysis.apply_chat_template(
        s['messages'], tokenize=False, add_generation_prompt=False
    )
    tokens = tokenizer_analysis(text, return_tensors='pt')['input_ids']
    lengths.append(tokens.shape[1])

lengths.sort()
n = len(lengths)

print('\n─── Token Length Distribution ───')
print(f'  P50 (median) : {lengths[int(n*0.50)]:,}')
print(f'  P90          : {lengths[int(n*0.90)]:,}')
print(f'  P95          : {lengths[int(n*0.95)]:,}')
print(f'  P99          : {lengths[int(n*0.99)]:,}')
print(f'  MAX          : {lengths[-1]:,}')
print(f'\n  MAX_SEQ_LEN  : {MAX_SEQ_LEN} (covers {sum(1 for l in lengths if l <= MAX_SEQ_LEN)/n*100:.1f}%)')

if sum(1 for l in lengths if l <= MAX_SEQ_LEN) / n < 0.95:
    print('⚠️  Coverage < 95% — consider increasing MAX_SEQ_LEN')
else:
    print('✅ Coverage ≥ 95% — MAX_SEQ_LEN is adequate')

del tokenizer_analysis
print('\n✅ Dataset analysis complete')

Loading tokenizer for analysis...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Total samples in file: 8,761
Using: 8,761 samples

─── Token Length Distribution ───
  P50 (median) : 1,720
  P90          : 3,292
  P95          : 3,634
  P99          : 4,254
  MAX          : 5,515

  MAX_SEQ_LEN  : 4096 (covers 98.4%)
✅ Coverage ≥ 95% — MAX_SEQ_LEN is adequate

✅ Dataset analysis complete


In [7]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Load model + LoRA
# ═══════════════════════════════════════════════════════════
import torch
from unsloth import FastLanguageModel

print('Loading base model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = BASE_MODEL,
    max_seq_length    = MAX_SEQ_LEN,
    dtype             = torch.float16,
    load_in_4bit      = True,
    trust_remote_code = True,
)

print('Applying LoRA adapters...')
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,      # MUST be 0.0
    target_modules             = LORA_TARGETS,
    bias                       = 'none',
    use_gradient_checkpointing = "unsloth",
)

# Trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\n✅ Model loaded')
print(f'   Trainable params : {trainable:,} ({trainable/total*100:.2f}%)')
print(f'   Total params     : {total:,}')
print(f'   LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')

Loading base model...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Applying LoRA adapters...


Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.



✅ Model loaded
   Trainable params : 29,933,568 (1.64%)
   Total params     : 1,830,055,936
   LoRA r=16, alpha=16, dropout=0.0


In [8]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Load + format dataset
# ═══════════════════════════════════════════════════════════
from datasets import Dataset
from unsloth.chat_templates import train_on_responses_only

print(f'Formatting {len(samples):,} samples...')

formatted = []
skipped = 0
for s in samples:
    try:
        text = tokenizer.apply_chat_template(
            s['messages'],
            tokenize=False,
            add_generation_prompt=False
        )
        formatted.append({'text': text})
    except Exception as e:
        skipped += 1

print(f'  Formatted : {len(formatted):,}')
if skipped > 0:
    print(f'  Skipped   : {skipped} (formatting errors)')

hf_dataset = Dataset.from_list(formatted)
split      = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_ds   = split['train']
eval_ds    = split['test']

print(f'  Train     : {len(train_ds):,}')
print(f'  Eval      : {len(eval_ds):,}')
print('\n✅ Dataset ready')

Formatting 8,761 samples...
  Formatted : 8,761
  Train     : 8,322
  Eval      : 439

✅ Dataset ready


In [ ]:
import os, warnings, logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DATASETS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import datasets, transformers
datasets.disable_progress_bars()
transformers.utils.logging.set_verbosity_error()
print("✅ noise suppressed")

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/qwen_bpp_v2_10k', ignore_errors=True)
shutil.rmtree('/kaggle/working/_savetest', ignore_errors=True)
print("✅ نُظّف القديم")

In [13]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Training (SFTConfig, eval per epoch, quiet)
# ═══════════════════════════════════════════════════════════
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
import time

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir=ADAPTERS_DIR,
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN, dataset_num_proc=1,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, lr_scheduler_type=LR_SCHEDULER, warmup_steps=WARMUP_STEPS,
        optim="adamw_8bit", fp16=True, bf16=False, max_grad_norm=MAX_GRAD_NORM,
        fp16_full_eval=True,                          # يسرّع الـ eval
        logging_steps=25,                             # طباعة loss كل 25 step
        eval_strategy="epoch",                        # eval كل epoch (أسرع)
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False,
        report_to="none", dataloader_num_workers=0,
        average_tokens_across_devices=False,
        disable_tqdm=False,                            # يخفي البار، يبقّي الأرقام
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

print('─── Training started (quiet mode) ───')
print(f'  {len(train_ds):,} train / {len(eval_ds):,} eval | {NUM_EPOCHS} epochs')
start = time.time()
r = trainer.train()
print(f'\n✅ Done {(time.time()-start)/3600:.2f}h | train_loss {r.training_loss:.4f}')

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/8322 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/439 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/8322 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/8322 [00:00<?, ? examples/s]

...

Map (num_proc=8):   0%|          | 0/439 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/439 [00:00<?, ? examples/s]

─── Training started (quiet mode) ───
  8,322 train / 439 eval | 1 epochs


Epoch,Training Loss,Validation Loss
1,0.786279,0.772125



✅ Done 6.11h | train_loss 0.8216


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Training
# Expected: ~6-8 hours on T4 for 10K samples, 3 epochs
# DO NOT restart session during training
# ═══════════════════════════════════════════════════════════
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only
import time

training_args = TrainingArguments(
    output_dir                    = ADAPTERS_DIR,
    num_train_epochs              = NUM_EPOCHS,
    per_device_train_batch_size   = BATCH_SIZE,
    per_device_eval_batch_size    = BATCH_SIZE,
    gradient_accumulation_steps   = GRAD_ACCUM,
    learning_rate                 = LR,
    lr_scheduler_type             = LR_SCHEDULER,
    warmup_steps                  = WARMUP_STEPS,   # NOT warmup_ratio
    optim                         = 'adamw_8bit',
    fp16                          = True,
    bf16                          = False,           # T4 doesn't support bf16
    max_grad_norm                 = MAX_GRAD_NORM,
    logging_steps                 = 25,
    eval_strategy                 = 'epoch',
    save_strategy                 = 'epoch',
    save_total_limit              = 3,
    load_best_model_at_end        = True,
    metric_for_best_model         = 'eval_loss',
    greater_is_better             = False,
    report_to                     = 'none',
    dataloader_num_workers        = 0,
    remove_unused_columns         = True,
    average_tokens_across_devices = False,
    
)

trainer = SFTTrainer(
    model              = model,
    train_dataset      = train_ds,
    eval_dataset       = eval_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 1,
    args               = training_args,
)

# Assistant-only loss masking
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part    = '<|im_start|>assistant\n',
)

print('─── Training started ───')
print(f'  Samples   : {len(train_ds):,} train / {len(eval_ds):,} eval')
print(f'  Epochs    : {NUM_EPOCHS}')
print(f'  Eff.Batch : {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LoRA r    : {LORA_R}')
print('  Expected  : ~6-8 hours on T4')
print('─' * 40)

start_time = time.time()
train_result = trainer.train()
elapsed = (time.time() - start_time) / 3600

print('\n─── Training complete ───')
print(f'  Time      : {elapsed:.2f} hours')
print(f'  Train loss: {train_result.training_loss:.4f}')
print('✅ Training done')

In [14]:
import os
for root, dirs, files in os.walk(ADAPTERS_DIR):
    for d in dirs:
        print(os.path.join(root, d))

/kaggle/working/qwen_bpp_v2_10k/adapters/checkpoint-1041


In [15]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Save adapters + merge model
# ═══════════════════════════════════════════════════════════
import os

# Save LoRA adapters
print('Saving LoRA adapters...')
model.save_pretrained(ADAPTERS_DIR)
tokenizer.save_pretrained(ADAPTERS_DIR)
print(f'✅ Adapters saved: {ADAPTERS_DIR}')

# Merge into full model (float16)
print('\nMerging model (this takes ~5 min)...')
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method='merged_16bit'
)

# Report sizes
def dir_size_gb(path):
    total = 0
    for f in os.scandir(path):
        total += f.stat().st_size
    return total / 1024**3

adapters_size = dir_size_gb(ADAPTERS_DIR)
merged_size   = dir_size_gb(MERGED_DIR)

print(f'\n✅ Merge complete')
print(f'   Adapters    : {adapters_size:.2f} GB  → {ADAPTERS_DIR}')
print(f'   Merged model: {merged_size:.2f} GB  → {MERGED_DIR}')

Saving LoRA adapters...
✅ Adapters saved: /kaggle/working/qwen_bpp_v2_10k/adapters

Merging model (this takes ~5 min)...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:15<00:00,  7.83s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:41<00:00, 20.51s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen_bpp_v2_10k/merged_model`

✅ Merge complete
   Adapters    : 0.12 GB  → /kaggle/working/qwen_bpp_v2_10k/adapters
   Merged model: 5.76 GB  → /kaggle/working/qwen_bpp_v2_10k/merged_model


In [40]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Inference test — sanity check (FIXED)
# ═══════════════════════════════════════════════════════════

import json, re, importlib.util
import torch

# ── Import FastLanguageModel (IMPORTANT) ───────────────────
from unsloth import FastLanguageModel


# ── Load Static Engine ──────────────────────────────────────
spec = importlib.util.spec_from_file_location('static_engine', static_path)
static_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(static_module)
analyze_code = static_module.analyze_code

print('✅ Static Engine loaded')


# ── Safe helpers ────────────────────────────────────────────
def format_issues(issues):
    if not issues:
        return 'None detected'
    lines = []
    for i in issues[:5]:
        sev  = i.get('severity', 'INFO')
        name = i.get('issue', i.get('type','?'))
        msg  = i.get('message', '')[:80]
        lines.append(f'  - [{sev}] {name}: {msg}')
    return '\n'.join(lines)


def run_inference(messages, max_new_tokens=1024):
    FastLanguageModel.for_inference(model)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,      # 🔥 deterministic for JSON stability
            do_sample=False,      # 🔥 IMPORTANT FIX
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = out[0][inputs.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def parse_output(raw):
    """Robust JSON parser"""
    text = re.sub(r'^```json\s*|^```\s*|\s*```$', '', raw.strip(), flags=re.MULTILINE)

    try:
        return json.loads(text), 'ok'
    except json.JSONDecodeError:
        # fallback: extract first valid JSON block
        brace_count = 0
        start = text.find('{')
        if start != -1:
            for i in range(start, len(text)):
                if text[i] == '{':
                    brace_count += 1
                elif text[i] == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        try:
                            return json.loads(text[start:i+1]), 'recovered'
                        except:
                            break

    return None, 'parse_failed'


# ── Test code ───────────────────────────────────────────────
TEST_CODE = '''
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    return db.execute(query)
'''

print('\n─── Running inference test ───')

# Step 1: Static Engine
result = analyze_code(TEST_CODE)

assistant_content = result['messages'][2]['content']
engine_data = json.loads(assistant_content)

scores = engine_data['scores']
labels = engine_data['labels']
issues = engine_data.get('issues', [])

print(f'⚙️  Static Engine:')
print(f'   Security        : {scores["security"]:.2f}/10  ({labels["security"]})')
print(f'   Complexity      : {scores["complexity"]:.2f}/10  ({labels["complexity"]})')
print(f'   Maintainability : {scores["maintainability"]:.2f}/10  ({labels["maintainability"]})')
print(f'   Issues          : {len(issues)}')


# Step 2: Build prompt
user_content = f"""Analyze this Python code:\n\n{TEST_CODE}\n
Engine Output:
  Security:        {scores['security']}/10 → {labels['security']}
  Complexity:      {scores['complexity']}/10 → {labels['complexity']}
  Maintainability: {scores['maintainability']}/10 → {labels['maintainability']}

Detected Issues:
{format_issues(issues)}"""

messages = [
    {
        'role': 'system',
        'content': (
            "You are a senior software engineer specializing in code quality, "
            "security, and static analysis. Return ONLY valid JSON."
        )
    },
    {'role': 'user', 'content': user_content}
]


# Step 3: Qwen inference
print('\n🤖 Qwen generating...')
raw_output = run_inference(messages)
parsed, status = parse_output(raw_output)

print(f'\n─── Qwen Output (status: {status}) ───')

if parsed:
    print('  ✅ Valid JSON')
    print(f'  Assessment       : {parsed.get("engine_assessment", "?")}')
    print(f'  Assessment reason : {parsed.get("assessment_reason", "?")}')
    print(f'  Reasoning        : {parsed.get("unified_reasoning", "?")[:150]}...')

    recs = parsed.get('recommendations', [])
    for i, rec in enumerate(recs, 1):
        print(f'  Rec {i}: {rec}')

    qwen_preds = parsed.get('qwen_score_predictions', {})
    if qwen_preds:
        print('\n  Qwen predictions (cross-check):')
        for k, v in qwen_preds.items():
            static_v = scores.get(k, 0)
            agreement = 1 - abs(static_v - v) / 10
            print(f'    {k:20s}: static={static_v:.2f}  qwen={v:.2f}  agreement={agreement:.2f}')
else:
    print('  ❌ Parse failed — raw output:')
    print(raw_output[:500])

print('\n✅ Inference test complete')

✅ Static Engine loaded

─── Running inference test ───
⚙️  Static Engine:
   Security        : 2.57/10  (Poor)
   Complexity      : 9.95/10  (Excellent)
   Maintainability : 6.90/10  (Acceptable)
   Issues          : 3

🤖 Qwen generating...


AttributeError: 'Qwen2Attention' object has no attribute 'apply_qkv'

In [23]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')

api = HfApi(token=hf_token)

# Create repo if not exists
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type='model',
    private=True,
    exist_ok=True
)

print(f'✅ Repo ready: {HF_REPO_ID}')

print(f'\nUploading merged model to {HF_REPO_ID}...')

api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=HF_REPO_ID,
    repo_type='model'
)

print('\n✅ Upload complete!')

✅ Repo ready: sarahjaradat2004/qwen-scorev-v2

Uploading merged model to sarahjaradat2004/qwen-scorev-v2...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Upload complete!


In [22]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('HF_TOKEN')

api = HfApi(token=token)
print(api.whoami())

{'type': 'user', 'id': '69e51e096020e65828a88396', 'name': 'sarahjaradat2004', 'fullname': 'SARAH JARADAT', 'email': 'szjaradat22@cit.just.edu.jo', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/hOtbVTGqra4XaJgPy2aUr.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'الي', 'role': 'write', 'createdAt': '2026-06-07T18:46:09.224Z'}}}


In [28]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — Final summary
# ═══════════════════════════════════════════════════════════
import json, os

# Load trainer state for loss history
state_path = os.path.join(ADAPTERS_DIR, 'trainer_state.json')
loss_history = []
if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    loss_history = [
        (e['epoch'], e.get('loss', '?'), e.get('eval_loss', '?'))
        for e in state.get('log_history', [])
        if 'eval_loss' in e
    ]

print('═' * 55)
print('         B++ FINE-TUNING — FINAL SUMMARY')
print('═' * 55)
print(f'  Run name      : {RUN_NAME}')
print(f'  Base model    : {BASE_MODEL}')
print(f'  Dataset       : {SAMPLE_SIZE:,} samples')
print(f'  LoRA          : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'  Epochs        : {NUM_EPOCHS}')
print(f'  Eff. batch    : {BATCH_SIZE * GRAD_ACCUM}')
print(f'  HF repo       : {HF_REPO_ID}')

if loss_history:
    print(f'\n  Loss per epoch:')
    for epoch, train_loss, eval_loss in loss_history:
        print(f'    Epoch {epoch:.0f}: train={train_loss}  eval={eval_loss}')

print(f'\n  Merged model  : {MERGED_DIR}')
print(f'  Adapters      : {ADAPTERS_DIR}')
print('═' * 55)
print('\n🚀 Next steps:')
print('  1. Build cross_validator.py')
print('  2. Run E1-E5 experiments')
print('  3. Build FastAPI endpoint (api.py)')
print('  4. Docker + Azure deployment')

═══════════════════════════════════════════════════════
         B++ FINE-TUNING — FINAL SUMMARY
═══════════════════════════════════════════════════════
  Run name      : qwen_bpp_v2_10k
  Base model    : Qwen/Qwen2.5-3B-Instruct
  Dataset       : 8,761 samples
  LoRA          : r=16, alpha=16, dropout=0.0
  Epochs        : 1
  Eff. batch    : 8
  HF repo       : sarahjaradat2004/qwen-scorev-v2

  Merged model  : /kaggle/working/qwen_bpp_v2_10k/merged_model
  Adapters      : /kaggle/working/qwen_bpp_v2_10k/adapters
═══════════════════════════════════════════════════════

🚀 Next steps:
  1. Build cross_validator.py
  2. Run E1-E5 experiments
  3. Build FastAPI endpoint (api.py)
  4. Docker + Azure deployment


In [30]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
api = HfApi()

# تحقق إن الموديل موجود فعلاً على HF
info = api.model_info('sarahjaradat2004/qwen-bpp-v2', token=hf_token)
print(f'✅ Model exists on HF: {info.modelId}')
print(f'   Last modified: {info.lastModified}')

✅ Model exists on HF: sarahjaradat2004/qwen-bpp-v2
   Last modified: 2026-05-22 17:56:44+00:00


In [36]:
model = AutoModelForCausalLM.from_pretrained(
    "sarahjaradat2004/qwen-bpp-v2",
    trust_remote_code=True,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    "sarahjaradat2004/qwen-bpp-v2"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]